# Monthly Inundation Record & Figure 2

Characterizes the delta-wide monthly AdDSWE inundation record (extent by confidence
class, seasonal cycle) and builds **Figure 2**.

- **Produces:** `LS_AdDSWE_monthly_summary_*.csv` (delta-wide monthly extent + QC stats) and Figure 2.
- **Inputs:** AdDSWE monthly products (Earth Engine assets) for area computation; the
  summary CSV for plotting (Dryad: `inundation/LS_AdDSWE_monthly_summary_*.csv`).

> Input/output paths are set within the notebook (to be pointed at a local Dryad copy).


In [ ]:
import os

# =============================== PATH CONFIGURATION ===============================
# Point DRYAD_ROOT at your local copy of the Dryad archive (doi:10.5061/dryad.msbcc2gct).
DRYAD_ROOT = r"."            # e.g. r"E:\Okavango\Data\For_Dryad"
OUTPUT_DIR = r"./outputs"    # local folder for derived CSVs / figures
os.makedirs(OUTPUT_DIR, exist_ok=True)


# DSWE Analysis

This code is used to analyze DSWE inundation assets generated using the 'Monthly Landsat DSWE Generator'script. The user provides folders to the DSWE and QC folders. It outputs a .csv of statistics including the DSWE pixel and area coverage by value, as well as QC statistics for each QC product. 

DSWE Methodology: Jones, J.W., 2019. Improved Automated Detection of Subpixel-Scale Inundation—Revised Dynamic Surface Water Extent (DSWE) Partial Surface Water Tests. Remote Sensing 11, 374. https://doi.org/10.3390/rs11040374

Landsat Collection 2: Earth Resources Observation and Science (EROS) Center. (2020). Landsat 8-9 Operational Land Imager / Thermal Infrared Sensor Level-2, Collection 2 [dataset]. U.S. Geological Survey. https://doi.org/10.5066/P9OGBGM6. Earth Resources Observation and Science (EROS) Center. (2020). Landsat 7 Enhanced Thematic Mapper Plus Level-2, Collection 2 [dataset]. U.S. Geological Survey. https://doi.org/10.5066/P9C7I13B. Earth Resources Observation and Science (EROS) Center. (2020). Landsat 4-5 Thematic Mapper Level-2, Collection 2 [dataset]. U.S. Geological Survey. https://doi.org/10.5066/P9IAXOVV.

Google Earth Engine: Gorelick, N., Hancher, M., Dixon, M., Ilyushchenko, S., Thau, D., Moore, R., 2017. Google Earth Engine: Planetary-scale geospatial analysis for everyone. Remote Sensing of Environment, Big Remotely Sensed Data: tools, applications and experiences 202, 18–27. https://doi.org/10.1016/j.rse.2017.06.031

Author: James (Huck) Rees, PhD Student, UC Santa Barbara Geography

Date: August 18, 2025

## Import packages

In [ ]:
import ee
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
from tqdm import tqdm

ee.Initialize()

## Initialize functions for Landsat DSWE

In [ ]:
def generate_monthly_ids(folder_prefix, prefix_label, start_year, start_month, end_year, end_month):
    start_date = datetime(start_year, start_month, 1)
    end_date = datetime(end_year, end_month, 1)
    image_ids = []

    while start_date <= end_date:
        year = start_date.year
        month = f"{start_date.month:02d}"
        img_id = f"{folder_prefix}/{prefix_label}_{year}_{month}"
        date_str = f"{year}-{month}"
        image_ids.append((img_id, date_str))
        start_date += relativedelta(months=1)

    return image_ids

def compute_dswe_stats(image_id, date_str, scale=30):
    stats = {'date': date_str}
    classes = [0, 1, 2, 3, 4]

    try:
        image = ee.Image(image_id).select('dswe')

        for cls in classes:
            count = image.eq(cls).reduceRegion(
                reducer=ee.Reducer.sum(),
                scale=scale,
                maxPixels=1e13
            ).get('dswe')
            area_km2 = ee.Number(count).multiply(scale * scale).divide(1e6)
            stats[f'count_{cls}'] = count.getInfo() if count else 0
            stats[f'area_km2_{cls}'] = area_km2.getInfo() if area_km2 else 0

        water_mask = image.gt(0)
        water_count = water_mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            scale=scale,
            maxPixels=1e13
        ).get('dswe')
        water_area = ee.Number(water_count).multiply(scale * scale).divide(1e6)

        stats['count_1to4'] = water_count.getInfo() if water_count else 0
        stats['area_km2_1to4'] = water_area.getInfo() if water_area else 0

        return stats

    except Exception:
        print(f"⚠️ DSWE image not found or failed to process: {image_id}")
        return None

def compute_qc_stats(qc_image_id, scale=30):
    try:
        image = ee.Image(qc_image_id).select('expansion_mask')
    except Exception:
        print(f"⚠️ QC image not found or failed to process: {qc_image_id}")
        return {}

    stats = {}

    for value in range(7):  # QC values 0–6
        mask = image.eq(value)
        count = mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            scale=scale,
            maxPixels=1e13
        ).get('expansion_mask')
        area_km2 = ee.Number(count).multiply(scale * scale).divide(1e6)
        stats[f'qc_count_{value}'] = count.getInfo() if count else 0
        stats[f'qc_area_km2_{value}'] = area_km2.getInfo() if area_km2 else 0

    # Total stats
    pixel_count = image.reduceRegion(
        reducer=ee.Reducer.count(),
        scale=scale,
        maxPixels=1e13
    ).get('expansion_mask')

    mean_value = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        scale=scale,
        maxPixels=1e13
    ).get('expansion_mask')

    total_area = ee.Number(pixel_count).multiply(scale * scale).divide(1e6)

    stats['qc_pixel_count'] = pixel_count.getInfo() if pixel_count else 0
    stats['qc_area_km2'] = total_area.getInfo() if total_area else 0
    stats['qc_avg_value'] = mean_value.getInfo() if mean_value else None

    return stats

def batch_process_LS_dswe_with_qc(dswe_folder, qc_folder, start_year, start_month, end_year, end_month, output_csv_path):
    image_entries = generate_monthly_ids(dswe_folder, 'DSWE', start_year, start_month, end_year, end_month)
    all_stats = []

    for dswe_id, date_str in image_entries:
        print(f"📅 Processing {date_str}...")
        dswe_stats = compute_dswe_stats(dswe_id, date_str)
        if dswe_stats:
            year, month = date_str.split('-')
            qc_id = f"{qc_folder}/QC_{year}_{month}"
            qc_stats = compute_qc_stats(qc_id)
            all_stats.append(dswe_stats | qc_stats)

    df = pd.DataFrame(all_stats)
    df['date'] = pd.to_datetime(df['date'], format='%Y-%m')
    df = df.sort_values(by='date')

    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)
    print(f"✅ CSV saved to: {output_csv_path}")

## Execute code for date range

In [ ]:
batch_process_LS_dswe_with_qc(
    dswe_folder='projects/ee-okavango/assets/water_masks/monthly_DSWE_Landsat_30m_v4/DSWE_Products',
    qc_folder='projects/ee-okavango/assets/water_masks/monthly_DSWE_Landsat_30m_v4/QC_Masks',
    start_year=1984,
    start_month=6,
    end_year=2025,
    end_month=12,
    output_csv_path=os.path.join(OUTPUT_DIR, "LS_AdDSWE_monthly_summary_01082026.csv")
)


## Plot inundated area by DSWE class by month (all years integrated)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_seasonal_data(csv_path):
    """
    Load and process monthly DSWE data from CSV.
    
    Parameters:
    -----------
    csv_path : str
        Path to CSV file with monthly DSWE summaries
    
    Returns:
    --------
    pd.DataFrame
        Monthly statistics with median and IQR for each confidence threshold
    """
    # Load raw data
    raw_df = pd.read_csv(csv_path)
    raw_df['date'] = pd.to_datetime(raw_df['date'])
    raw_df['month'] = raw_df['date'].dt.month
    
    # Calculate combined areas
    raw_df['area_4'] = raw_df['area_km2_4']
    raw_df['area_34'] = raw_df['area_km2_3'] + raw_df['area_km2_4']
    raw_df['area_234'] = raw_df['area_km2_2'] + raw_df['area_km2_3'] + raw_df['area_km2_4']
    raw_df['area_1234'] = raw_df['area_km2_1to4']
    
    # Group by month and calculate statistics
    df = raw_df.groupby('month').agg({
        'area_4': ['median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)],
        'area_34': ['median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75), 'count'],
        'area_234': ['median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)],
        'area_1234': ['median', lambda x: np.percentile(x, 25), lambda x: np.percentile(x, 75)]
    }).reset_index()
    
    # Flatten column names
    df.columns = ['month', 'median_4', 'q25_4', 'q75_4',
                  'median_34', 'q25_34', 'q75_34', 'n_years',
                  'median_234', 'q25_234', 'q75_234',
                  'median_1234', 'q25_1234', 'q75_1234']
    
    return df

def plot_seasonal_flood_pulse(df):
    """
    Create four-panel plot of seasonal flood pulse.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Processed monthly statistics from load_seasonal_data()
    """
    # Set font to Arial size 8 for all text
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.size'] = 8
    
    fig, axes = plt.subplots(4, 1, figsize=(6.5, 8))
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    # Plot 1: Class 4 only (High Confidence Open Water)
    ax0 = axes[0]
    ax0.plot(df['month'], df['median_4'], 'o-', color='#004F8F', linewidth=2, markersize=6, label='Median')
    ax0.fill_between(df['month'], df['q25_4'], df['q75_4'], alpha=0.3, color='#004F8F', label='IQR')
    ax0.set_ylabel('Area (km²)', fontsize=8)
    ax0.set_title('High confidence open water (class 4)', fontsize=8, fontweight='bold')
    ax0.set_xticks(range(1, 13))
    ax0.set_xticklabels(month_names, fontsize=8)
    ax0.tick_params(axis='y', labelsize=8)
    ax0.grid(True, alpha=0.3)
    ax0.legend(loc='upper right', fontsize=8)
    ax0.set_xlim(0.5, 12.5)
    
    # Plot 2: Classes 3+4 (Moderate Confidence Water)
    ax1 = axes[1]
    ax1.plot(df['month'], df['median_34'], 'o-', color='#3F8FFF', linewidth=2, markersize=6, label='Median')
    ax1.fill_between(df['month'], df['q25_34'], df['q75_34'], alpha=0.3, color='#3F8FFF', label='IQR')
    ax1.set_ylabel('Area (km²)', fontsize=8)
    ax1.set_title('Combined classes 3+4', fontsize=8, fontweight='bold')
    ax1.set_xticks(range(1, 13))
    ax1.set_xticklabels(month_names, fontsize=8)
    ax1.tick_params(axis='y', labelsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='upper right', fontsize=8)
    ax1.set_xlim(0.5, 12.5)
    
    # Plot 3: Classes 2+3+4 (Potential Wetland)
    ax2 = axes[2]
    ax2.plot(df['month'], df['median_234'], 'o-', color='#7FBFFF', linewidth=2, markersize=6, label='Median')
    ax2.fill_between(df['month'], df['q25_234'], df['q75_234'], alpha=0.3, color='#7FBFFF', label='IQR')
    ax2.set_ylabel('Area (km²)', fontsize=8)
    ax2.set_title('Combined classes 2+3+4', fontsize=8, fontweight='bold')
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels(month_names, fontsize=8)
    ax2.tick_params(axis='y', labelsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper right', fontsize=8)
    ax2.set_xlim(0.5, 12.5)
    
    # Plot 4: Classes 1+2+3+4 (Low Confidence Water)
    ax3 = axes[3]
    ax3.plot(df['month'], df['median_1234'], 'o-', color='#BFDFFF', linewidth=2, markersize=6, label='Median')
    ax3.fill_between(df['month'], df['q25_1234'], df['q75_1234'], alpha=0.3, color='#BFDFFF', label='IQR')
    ax3.set_xlabel('Month', fontsize=8)
    ax3.set_ylabel('Area (km²)', fontsize=8)
    ax3.set_title('Combined classes 1+2+3+4', fontsize=8, fontweight='bold')
    ax3.set_xticks(range(1, 13))
    ax3.set_xticklabels(month_names, fontsize=8)
    ax3.tick_params(axis='y', labelsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='upper right', fontsize=8)
    ax3.set_xlim(0.5, 12.5)
    
    plt.tight_layout()
    plt.show()

# Usage:
csv_path = os.path.join(DRYAD_ROOT, "inundation", "LS_AdDSWE_monthly_summary_01082026.csv")
df = load_seasonal_data(csv_path)
plot_seasonal_flood_pulse(df)

## Inundation extent calculations

In [ ]:
import pandas as pd
import numpy as np

def calculate_results_paragraph_stats(csv_path):
    """
    Calculate all missing statistics for the results paragraph.
    """
    # Load data
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    # Compute combined areas
    df['area_1234'] = df['area_km2_1to4']
    df['area_34'] = df['area_km2_3'] + df['area_km2_4']

    month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April',
                   5: 'May', 6: 'June', 7: 'July', 8: 'August',
                   9: 'September', 10: 'October', 11: 'November', 12: 'December'}

    print("=" * 60)
    print("RESULTS PARAGRAPH STATISTICS")
    print("=" * 60)

    # ── Total record count ────────────────────────────────────────
    print(f"\nTotal monthly composites: {len(df)}")

    # ── Annual max/min for total inundation (classes 1–4) ─────────
    annual_max_1234 = df.groupby('year')['area_1234'].max()
    annual_min_1234 = df.groupby('year')['area_1234'].min()

    # Absolute max and min across all years
    abs_max_1234_val = annual_max_1234.max()
    abs_max_1234_year = annual_max_1234.idxmax()
    abs_max_1234_month = df.loc[df['area_1234'].idxmax(), 'month']

    abs_min_1234_val = annual_min_1234.min()
    abs_min_1234_year = annual_min_1234.idxmin()
    abs_min_1234_month = df.loc[df['area_1234'].idxmin(), 'month']

    mean_annual_max_1234 = annual_max_1234.mean()
    mean_annual_min_1234 = annual_min_1234.mean()

    print("\n── Total inundation (classes 1–4) ──────────────────────")
    print(f"  Maximum extent:        {abs_max_1234_val:.1f} km²  "
          f"({month_names[abs_max_1234_month]} {abs_max_1234_year})")
    print(f"  Minimum extent:        {abs_min_1234_val:.1f} km²  "
          f"({month_names[abs_min_1234_month]} {abs_min_1234_year})")
    print(f"  Mean annual maximum:   {mean_annual_max_1234:.1f} km²")
    print(f"  Mean annual minimum:   {mean_annual_min_1234:.1f} km²")

    # ── Annual max/min for open water (classes 3–4) ───────────────
    annual_max_34 = df.groupby('year')['area_34'].max()
    annual_min_34 = df.groupby('year')['area_34'].min()

    abs_max_34_val = annual_max_34.max()
    abs_max_34_year = annual_max_34.idxmax()
    abs_max_34_month = df.loc[df['area_34'].idxmax(), 'month']

    abs_min_34_val = annual_min_34.min()
    abs_min_34_year = annual_min_34.idxmin()
    abs_min_34_month = df.loc[df['area_34'].idxmin(), 'month']

    mean_annual_max_34 = annual_max_34.mean()
    mean_annual_min_34 = annual_min_34.mean()

    print("\n── Open water (classes 3–4) ────────────────────────────")
    print(f"  Maximum extent:        {abs_max_34_val:.1f} km²  "
          f"({month_names[abs_max_34_month]} {abs_max_34_year})")
    print(f"  Minimum extent:        {abs_min_34_val:.1f} km²  "
          f"({month_names[abs_min_34_month]} {abs_min_34_year})")
    print(f"  Mean annual maximum:   {mean_annual_max_34:.1f} km²")
    print(f"  Mean annual minimum:   {mean_annual_min_34:.1f} km²")

    # ── Open water as proportion of total inundation ──────────────
    df['ow_fraction'] = df['area_34'] / df['area_1234']
    mean_ow_pct = df['ow_fraction'].mean() * 100

    # Monthly climatology of open water fraction
    monthly_ow_frac = df.groupby('month')['ow_fraction'].median() * 100
    peak_month = monthly_ow_frac.idxmax()
    trough_month = monthly_ow_frac.idxmin()

    print("\n── Open water fraction of total inundation ─────────────")
    print(f"  Mean across full record:   {mean_ow_pct:.1f}%")
    print(f"  Highest monthly fraction:  {monthly_ow_frac[peak_month]:.1f}%  "
          f"({month_names[peak_month]})")
    print(f"  Lowest monthly fraction:   {monthly_ow_frac[trough_month]:.1f}%  "
          f"({month_names[trough_month]})")

    print("\n── Monthly climatology: open water fraction by month ───")
    for m, val in monthly_ow_frac.items():
        print(f"  {month_names[m]:<12}  {val:.1f}%")

    # ── Seasonal timing: peak months for open water vs total ──────
    monthly_clim_34   = df.groupby('month')['area_34'].median()
    monthly_clim_1234 = df.groupby('month')['area_1234'].median()

    peak_ow_month    = monthly_clim_34.idxmax()
    peak_total_month = monthly_clim_1234.idxmax()

    print("\n── Seasonal timing (climatological medians) ────────────")
    print(f"  Open water peak month:        {month_names[peak_ow_month]} "
          f"(median {monthly_clim_34[peak_ow_month]:.1f} km²)")
    print(f"  Total inundation peak month:  {month_names[peak_total_month]} "
          f"(median {monthly_clim_1234[peak_total_month]:.1f} km²)")
    print(f"  Lag (total minus open water): "
          f"{peak_total_month - peak_ow_month} month(s)")

    # ── Top 3 months for each for manuscript context ──────────────
    top3_ow    = monthly_clim_34.nlargest(3)
    top3_total = monthly_clim_1234.nlargest(3)

    print("\n── Top 3 months: open water (classes 3–4) ─────────────")
    for m, val in top3_ow.items():
        print(f"  {month_names[m]:<12}  {val:.1f} km²")

    print("\n── Top 3 months: total inundation (classes 1–4) ────────")
    for m, val in top3_total.items():
        print(f"  {month_names[m]:<12}  {val:.1f} km²")

    print("\n" + "=" * 60)


# Usage:
csv_path = os.path.join(DRYAD_ROOT, "inundation", "LS_AdDSWE_monthly_summary_01082026.csv")
calculate_results_paragraph_stats(csv_path)